# 📊 Excel Mastery with Python: pandas, openpyxl & xlsxwriter

**A hands-on tutorial + exercise workbook for going from "I can open a spreadsheet" to
"I can automate messy, real-world Excel workflows in Python."**

### How this notebook works
Every topic follows the same three-step rhythm:

1. **Concept** – a short explanation of *what* the technique is and *why* it matters on the job.
2. **Worked Example** – complete, runnable code with comments explaining *why*, not just *what*.
3. **Practice Challenge** – a `# TODO:` exercise for you to finish, with hints. Your code should
   assign its result to the variable named in the TODO so you can check your work.

> 💡 A **Solutions Appendix** is at the very end of the notebook. Try each challenge yourself first —
> the learning happens in the struggle, not in reading the answer.

### Libraries covered
| Library | What it's best at |
|---|---|
| `pandas` | Fast, vectorized data cleaning, joining, and aggregation |
| `openpyxl` | Reading/writing native `.xlsx` files, cell-level formatting, formulas, charts |
| `xlsxwriter` | Write-only, but excellent for polished formatting and charts in one pass |

### The dataset
Rather than asking you to hunt down a file, this notebook **generates its own messy,
realistic dataset** the first time you run it — a sales export (`RawExport`) with all the warts
of a real system dump (banner rows, inconsistent casing, missing values, mixed date formats),
plus two clean reference/master tables (`Customers`, `Products`). This mirrors a very common
real-world pattern: **transactional exports are messy, reference/master data is usually cleaner.**

Run the **Setup** cell below once — everything after it reads from the file it creates, just like
a real analysis would.

## ⚙️ Setup: Generate a realistic messy workbook

We build a workbook called **`company_data.xlsx`** with three sheets:

- **`RawExport`** — a messy transactional sales log, complete with a banner/title row, inconsistent
  column names, mixed date formats, missing quantities, and inconsistent text casing.
- **`Customers`** — a clean(ish) customer master/reference table.
- **`Products`** — a clean product master/reference table, including unit cost (used later for our
  VLOOKUP/XLOOKUP-style lookups).

Run this cell once. Everything downstream reads this file from disk, exactly like a real workflow.

In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
from openpyxl import load_workbook

# Seed both random number generators so the "messy" data is identical every run —
# reproducibility matters when you're teaching (or debugging!) a pipeline.
np.random.seed(42)
random.seed(42)

# ---------------------------------------------------------------
# 1) PRODUCTS master table (reference data — usually clean)
# ---------------------------------------------------------------
products = pd.DataFrame({
    "ProductCode": [f"P{str(i).zfill(3)}" for i in range(1, 16)],
    "ProductName": [
        "Laptop Pro 14", "Wireless Mouse", "USB-C Hub", "27in Monitor", "Mechanical Keyboard",
        "Office Chair", "Standing Desk", "Filing Cabinet", "Desk Lamp", "Bookshelf",
        "Printer Paper (500ct)", "Sticky Notes Pack", "Whiteboard Markers", "Stapler", "Binder Clips",
    ],
    "Category": ["Electronics"] * 5 + ["Furniture"] * 5 + ["Office Supplies"] * 5,
    "UnitCost": [1299.00, 24.50, 39.99, 249.00, 89.00,
                 189.00, 349.00, 159.00, 45.00, 129.00,
                 6.75, 3.20, 4.50, 8.10, 2.25],
})

# ---------------------------------------------------------------
# 2) CUSTOMERS master table (reference data)
# ---------------------------------------------------------------
prefixes = ["Kivu", "Nyanza", "Savanna", "Baobab", "Rift Valley", "Lakeview", "Sahel", "Sunrise",
            "Highland", "Delta", "Horizon", "Cedar", "Amber", "Falcon", "Granite", "Ivory", "Jade",
            "Kestrel", "Meridian", "Nova", "Opal", "Prairie", "Quartz", "Ridge", "Summit"]
suffixes = ["Traders Ltd", "Tech Solutions", "Holdings", "Logistics", "Consulting Group",
            "Enterprises", "Retailers", "Manufacturing", "Services", "Partners"]
segments = ["Enterprise", "SMB", "Startup", "Government"]
countries = ["Rwanda", "Kenya", "Uganda", "Tanzania", "Nigeria"]

customers = pd.DataFrame({
    "CustID": [f"C{str(i).zfill(3)}" for i in range(1, 26)],
    "CustomerName": [f"{p} {random.choice(suffixes)}" for p in prefixes],
    "Segment": [random.choice(segments) for _ in range(25)],
    "Country": [random.choice(countries) for _ in range(25)],
})

# ---------------------------------------------------------------
# 3) RAW SALES export (messy transactional data)
# ---------------------------------------------------------------
n_orders = 150
order_ids = [f"ORD-{1000 + i}" for i in range(n_orders)]
order_ids[10] = order_ids[5]     # inject a couple of duplicate Order IDs...
order_ids[80] = order_ids[81]    # ...like a real system re-sending a record

reps_clean = ["Aline U.", "Eric N.", "Grace M.", "Patrick K.", "Diane I.", "Jean B.", "Claudine R.", "Yves T."]
regions_clean = ["North", "South", "East", "West"]
note_pool = ["", "Rush order", "Gift wrap requested", "Bulk discount applied", np.nan, "Follow up needed"]

def messy_region(r):
    return random.choice([r, r.upper(), r.lower(), f" {r}", f"{r} "])

def messy_rep(r):
    if random.random() < 0.05:
        return np.nan  # ~5% of orders are missing a sales rep, just like real CRMs
    return random.choice([r, f" {r}", f"{r}  ", r.upper()])

def messy_date(d):
    roll = random.random()
    if roll < 0.03:
        return random.choice(["TBD", "N/A", ""])  # a few genuinely broken date entries
    fmt = random.choice(["%m/%d/%Y", "%d-%b-%Y", "%Y-%m-%d"])  # inconsistent formats per row
    return d.strftime(fmt)

date_start = datetime(2026, 1, 1)
rows = []
for i in range(n_orders):
    d = date_start + timedelta(days=random.randint(0, 180))
    qty = random.randint(1, 10)
    if random.random() < 0.05:
        qty = np.nan  # ~5% missing quantity
    rows.append({
        " OrderID": order_ids[i],                                   # leading space in header
        "Order Date ": messy_date(d),                                # trailing space in header
        "CustID ": random.choice(customers["CustID"].tolist()),      # trailing space
        " ProductCode": random.choice(products["ProductCode"].tolist()),
        "Qty": qty,
        " Region ": messy_region(random.choice(regions_clean)),
        "SalesRep": messy_rep(random.choice(reps_clean)),
        "Notes": random.choice(note_pool),
    })

sales_raw = pd.DataFrame(rows)

# ---------------------------------------------------------------
# 4) Write everything to one workbook, with a banner row above the
#    real header on RawExport — a very common export artifact.
# ---------------------------------------------------------------
with pd.ExcelWriter("company_data.xlsx", engine="openpyxl") as writer:
    # startrow=2 leaves two blank rows above the header (rows 1-2), header lands on row 3
    sales_raw.to_excel(writer, sheet_name="RawExport", index=False, startrow=2)
    customers.to_excel(writer, sheet_name="Customers", index=False)
    products.to_excel(writer, sheet_name="Products", index=False)

wb = load_workbook("company_data.xlsx")
ws = wb["RawExport"]
ws["A1"] = "Company Sales Export -- Generated by FinanceSys -- DO NOT EDIT MANUALLY"
wb.save("company_data.xlsx")

print("Created company_data.xlsx with sheets:", wb.sheetnames)
print(f"RawExport: {len(sales_raw)} rows (messy) | Customers: {len(customers)} rows | Products: {len(products)} rows")

Created company_data.xlsx with sheets: ['RawExport', 'Customers', 'Products']
RawExport: 150 rows (messy) | Customers: 25 rows | Products: 15 rows


---
# Module 1: Loading & Inspection

Before you can clean anything, you have to get it into pandas correctly — and correctly reading a
real-world Excel export is harder than `pd.read_excel(file)` more often than beginners expect.
This module covers multi-sheet workbooks, banner/junk rows, bad headers, and how to quickly profile
what you've loaded.

## 1.1 Multi-sheet workbooks

**Concept:** A single `.xlsx` file can contain many sheets. `pandas.read_excel()` reads **one sheet
at a time** by default (the first one) — you must tell it which sheet you want via `sheet_name`.
Passing `sheet_name=None` reads **every** sheet at once into a dictionary of DataFrames, which is
great for a first look at an unfamiliar workbook. For metadata-only questions (sheet names, without
loading any data), `openpyxl.load_workbook(..., read_only=True)` is faster and lighter.

**Why it matters on the job:** Real workbooks are rarely single-sheet. Finance exports, CRM dumps,
and multi-tab trackers are the norm — the first step of almost any analysis is figuring out what's
actually inside the file before you touch a single row.

In [2]:
import pandas as pd
from openpyxl import load_workbook

# --- Option A: peek at structure without loading any data (fast, metadata only) ---
wb = load_workbook("company_data.xlsx", read_only=True)
print("Sheet names:", wb.sheetnames)
for name in wb.sheetnames:
    ws = wb[name]
    print(f"  {name}: {ws.max_row} rows x {ws.max_column} cols")
wb.close()

# --- Option B: load every sheet into a dict of DataFrames in one call ---
all_sheets = pd.read_excel("company_data.xlsx", sheet_name=None)
print("\nLoaded sheets:", list(all_sheets.keys()))
for name, df in all_sheets.items():
    print(f"  {name}: shape={df.shape}")

# Notice RawExport looks wrong right now -- its "header" is actually the banner text,
# because pandas blindly used row 1 as the header. We fix that in section 1.2.
all_sheets["RawExport"].head(3)

Sheet names: ['RawExport', 'Customers', 'Products']
  RawExport: 153 rows x 8 cols
  Customers: 26 rows x 4 cols
  Products: 16 rows x 4 cols

Loaded sheets: ['RawExport', 'Customers', 'Products']
  RawExport: shape=(152, 8)
  Customers: shape=(25, 4)
  Products: shape=(15, 4)


,Company Sales Export -- Generated by FinanceSys -- DO NOT EDIT MANUALLY,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OrderID,Order Date,CustID,ProductCode,Qty,Region,SalesRep,Notes
2,ORD-1000,02/26/2026,C002,P013,6,East,Diane I.,Rush order


### 🧩 Practice 1.1
Load **only** the `Products` sheet (not all sheets) into a variable called `df_products_check`,
using the `sheet_name` argument. Then print its shape and its column names.

**Hints:**
- `sheet_name` accepts a string (one sheet), a list of strings (several sheets), or `None` (all sheets).
- `df.columns` gives you the column names; `df.shape` gives you `(rows, cols)`.

In [3]:
# TODO: read only the 'Products' sheet from company_data.xlsx into df_products_check
df_products_check = None  # <- replace this

# TODO: print df_products_check.shape and df_products_check.columns

## 1.2 Skipped rows & bad headers

**Concept:** Real exports often have banner/title rows, blank separator rows, or merged "report
title" cells sitting above the actual header row. If you read these naively, pandas will treat the
banner text as your column names and everything breaks downstream. `skiprows=N` tells pandas to
ignore the first `N` physical rows before looking for a header.

**Why it matters on the job:** This is probably the single most common reason a "simple"
`read_excel()` call throws confusing errors or silently produces garbage columns like `Unnamed: 0`.
Always **look at the raw file first** (or print the first few rows with `header=None`) before
trusting the default header detection.

In [4]:
# First, diagnose the problem: read with header=None to see raw rows exactly as they are
raw_peek = pd.read_excel("company_data.xlsx", sheet_name="RawExport", header=None, nrows=5)
print("Raw rows before fixing:")
print(raw_peek)

# Now read it correctly: skip the banner row + blank row, let row 3 become the header
df_sales_raw = pd.read_excel("company_data.xlsx", sheet_name="RawExport", skiprows=2)

print("\nColumns as pandas sees them now (note the stray whitespace!):")
print(list(df_sales_raw.columns))
df_sales_raw.head()

Raw rows before fixing:
                                                   0            1        2  \
0  Company Sales Export -- Generated by FinanceSy...          NaN      NaN   
1                                                NaN          NaN      NaN   
2                                            OrderID  Order Date   CustID    
3                                           ORD-1000   02/26/2026     C002   
4                                           ORD-1001  17-Jun-2026     C005   

              3    4         5           6           7  
0           NaN  NaN       NaN         NaN         NaN  
1           NaN  NaN       NaN         NaN         NaN  
2   ProductCode  Qty   Region     SalesRep       Notes  
3          P013    6      East  Diane I.    Rush order  
4          P005    8     SOUTH    DIANE I.         NaN  

Columns as pandas sees them now (note the stray whitespace!):
[' OrderID', 'Order Date ', 'CustID ', ' ProductCode', 'Qty', ' Region ', 'SalesRep', 'Notes']


,OrderID,Order Date,CustID,ProductCode,Qty,Region,SalesRep,Notes
0,ORD-1000,02/26/2026,C002,P013,6.0,East,Diane I.,Rush order
1,ORD-1001,17-Jun-2026,C005,P005,8.0,SOUTH,DIANE I.,NaN
2,ORD-1002,2026-04-13,C016,P002,6.0,North,GRACE M.,NaN
3,ORD-1003,2026-01-17,C009,P009,7.0,North,Diane I.,NaN
4,ORD-1004,NaN,C024,P005,7.0,South,Eric N.,Follow up needed


### 🧩 Practice 1.2
The columns loaded above still contain leading/trailing whitespace (e.g. `" OrderID"`,
`"Order Date "`). Using `header=None` and `nrows`, confirm for yourself which **physical row number**
the real header sits on if you did **not** already know it was row 3. Store your discovered
`skiprows` value as an integer in `discovered_skiprows`.

**Hints:**
- Read a handful of rows with `header=None` and eyeball where real column-like text starts.
- `skiprows` should equal the number of *rows above* the header, not the header's row number itself.

In [5]:
# TODO: inspect the first several raw rows of 'RawExport' with header=None
# TODO: set discovered_skiprows to the correct integer based on what you see
discovered_skiprows = None  # <- replace this

## 1.3 Inspecting metadata

**Concept:** Before cleaning, always profile what you've got: shape, dtypes, a statistical summary,
and a quick look at both ends of the data. `.info()` is usually your very first move — it shows
row count, non-null counts per column, and inferred dtypes all in one call.

**Why it matters on the job:** Five seconds of `.info()` and `.describe()` routinely saves you from
building an entire analysis on a column pandas silently mis-typed (e.g. a date column read in as
plain text, or a "quantity" column that's actually `object` dtype because of a stray missing value).

In [6]:
print("Shape:", df_sales_raw.shape)
print()
df_sales_raw.info()
print()
print("Numeric summary:")
print(df_sales_raw.describe(include="all").T[["count", "unique", "top", "freq"]] if hasattr(df_sales_raw.describe(include="all"), "T") else None)

Shape: (150, 8)

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0    OrderID      150 non-null    str    
 1   Order Date    146 non-null    str    
 2   CustID        150 non-null    str    
 3    ProductCode  150 non-null    str    
 4   Qty           139 non-null    float64
 5    Region       150 non-null    str    
 6   SalesRep      143 non-null    str    
 7   Notes         103 non-null    str    
dtypes: float64(1), str(7)
memory usage: 9.5 KB

Numeric summary:
              count unique                  top freq
 OrderID        150    148             ORD-1005    2
Order Date      146    129          17-Jun-2026    3
CustID          150     25                 C002   11
 ProductCode    150     15                 P013   15
Qty           139.0    NaN                  NaN  NaN
 Region         150     20                 East   13
SalesRep        143    

### 🧩 Practice 1.3
Print the **dtype** of the `"Qty"` column and the **number of unique values** in the `" Region "`
column (mind the stray spaces in that column name!). Store the dtype in `qty_dtype` and the unique
count in `region_unique_count`.

**Hints:**
- `df[col].dtype` gives a single column's dtype.
- `df[col].nunique()` counts distinct non-null values.
- Copy-paste the column name exactly, including spaces, or use `df.columns` to find it precisely.

In [7]:
# TODO: qty_dtype = df_sales_raw["Qty"].dtype
qty_dtype = None  # <- replace this

# TODO: region_unique_count = df_sales_raw[<region column name>].nunique()
region_unique_count = None  # <- replace this

## 1.4 Missing data & duplicates

**Concept:** `.isna().sum()` gives you a per-column count of missing values in one line.
`.duplicated()` flags rows that are exact repeats of an earlier row (or, with `subset=`, repeats on
just the columns you care about — e.g. a duplicate Order ID even if other fields differ slightly).

**Why it matters on the job:** Missing values and duplicate records are two of the most common
silent sources of wrong totals in business reporting. Catching them at load time — before you sum
or join anything — is a habit that prevents a lot of "why don't these numbers match Excel" emails.

In [8]:
print("Missing values per column:")
print(df_sales_raw.isna().sum())

print("\nFully duplicated rows:", df_sales_raw.duplicated().sum())

# A more realistic check for THIS dataset: duplicate Order IDs (even if other fields drifted)
order_id_col = " OrderID"
dupe_order_ids = df_sales_raw[df_sales_raw.duplicated(subset=[order_id_col], keep=False)]
print(f"\nRows sharing a duplicated {order_id_col.strip()!r}: {len(dupe_order_ids)}")
dupe_order_ids.sort_values(order_id_col)

Missing values per column:
 OrderID         0
Order Date       4
CustID           0
 ProductCode     0
Qty             11
 Region          0
SalesRep         7
Notes           47
dtype: int64

Fully duplicated rows: 0

Rows sharing a duplicated 'OrderID': 4


,OrderID,Order Date,CustID,ProductCode,Qty,Region,SalesRep,Notes
5,ORD-1005,05/10/2026,C018,P013,10.0,North,Jean B.,Gift wrap requested
10,ORD-1005,27-May-2026,C007,P002,10.0,North,JEAN B.,Follow up needed
80,ORD-1081,03/28/2026,C022,P012,2.0,West,JEAN B.,NaN
81,ORD-1081,2026-03-09,C022,P005,9.0,west,ERIC N.,NaN


### 🧩 Practice 1.4
Calculate what **percentage** of rows in `df_sales_raw` have a missing `"Qty"` value. Store the
result (a float, e.g. `4.67` for 4.67%) in `pct_missing_qty`.

**Hints:**
- `df[col].isna().mean()` gives the *fraction* missing directly (mean of a boolean Series).
- Multiply by 100 to convert a fraction to a percentage.

In [9]:
# TODO: compute the percentage of missing "Qty" values
pct_missing_qty = None  # <- replace this

---
# Module 2: Cleaning & Transformation

Now that `df_sales_raw` is loaded correctly, it's time to make it *usable*: sane column names,
clean text, real dates, correct numeric types, and a couple of calculated columns. By the end of
this module we'll have one canonical, clean DataFrame — `df_sales` — that every later module
builds on.

## 2.1 Renaming columns

**Concept:** `df.columns = [...]` overwrites all column names at once (fast, but you must supply
every name in order). `df.rename(columns={...})` is safer when you only want to fix a few names,
since it works off a mapping and won't silently misalign if the column order ever changes.
A very common pattern is normalizing every column to `snake_case` with `str.strip().str.lower()`
in one shot before doing anything else.

**Why it matters on the job:** Downstream code (`df["order_date"]`) is far more reliable than
constantly re-typing `df[" Order Date "]` with the exact whitespace from a messy export — and
`snake_case` avoids a whole class of typo bugs.

In [10]:
# Normalize every column name in one vectorized pass: strip whitespace, lowercase, spaces -> underscores
df_sales = df_sales_raw.copy()  # never mutate the original loaded data -- keep a raw reference around
df_sales.columns = (
    df_sales.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
)

# A few names still don't match our target schema exactly (no space to auto-replace with an
# underscore) -- fix those explicitly with rename()
df_sales = df_sales.rename(columns={
    "custid": "customer_id",
    "productcode": "product_code",
    "orderid": "order_id",
    "salesrep": "sales_rep",
})

print(df_sales.columns.tolist())
df_sales.head()

['order_id', 'order_date', 'customer_id', 'product_code', 'qty', 'region', 'sales_rep', 'notes']


,order_id,order_date,customer_id,product_code,qty,region,sales_rep,notes
0,ORD-1000,02/26/2026,C002,P013,6.0,East,Diane I.,Rush order
1,ORD-1001,17-Jun-2026,C005,P005,8.0,SOUTH,DIANE I.,NaN
2,ORD-1002,2026-04-13,C016,P002,6.0,North,GRACE M.,NaN
3,ORD-1003,2026-01-17,C009,P009,7.0,North,Diane I.,NaN
4,ORD-1004,NaN,C024,P005,7.0,South,Eric N.,Follow up needed


### 🧩 Practice 2.1
Starting from `df_sales`, use `.rename()` to rename the `"notes"` column to `"order_notes"`
(a clearer, more specific name now that this table has multiple entities' data mixed in). Save the
result into a **new** variable `df_sales_practice` — don't overwrite `df_sales` itself, since later
sections build on the original column names.

**Hints:**
- `df.rename(columns={"old_name": "new_name"})` returns a new DataFrame by default — assign it to
  `df_sales_practice` rather than using `inplace=True`, so `df_sales` stays untouched.
- Confirm your work with `df_sales_practice.columns.tolist()`.

In [11]:
# TODO: df_sales_practice = df_sales.rename(columns={...})
df_sales_practice = None  # <- replace this

## 2.2 String manipulation

**Concept:** The `.str` accessor applies string methods to an entire column at once (vectorized —
much faster than a Python loop). Common cleaning moves: `.str.strip()` (remove leading/trailing
whitespace), `.str.title()` / `.str.upper()` / `.str.lower()` (normalize casing), and
`.fillna()` for text columns where "missing" should become an explicit label rather than `NaN`.

**Why it matters on the job:** `"East"`, `" east "`, and `"EAST "` all mean the same thing to a
human but are three different groups to `.groupby()` unless you normalize them first. This is the
single most common cause of an "extra category" nobody can explain in a summary report.

In [12]:
# Region: strip stray whitespace, then Title Case it so "EAST", " east ", "East " all collapse together
df_sales["region"] = df_sales["region"].str.strip().str.title()

# Sales rep: same cleanup, plus an explicit label for missing values (better than a silent NaN
# that quietly drops out of groupby() results later). Already renamed to sales_rep in section 2.1.
df_sales["sales_rep"] = df_sales["sales_rep"].str.strip().str.title()
df_sales["sales_rep"] = df_sales["sales_rep"].fillna("Unassigned")

# Notes: empty strings and NaN both mean "no note" here -- make that consistent
df_sales["notes"] = df_sales["notes"].replace("", pd.NA).fillna("No note")

print(df_sales["region"].value_counts())
print()
print(df_sales["sales_rep"].value_counts())

region
East     49
West     41
South    30
North    30
Name: count, dtype: int64

sales_rep
Diane I.       24
Yves T.        19
Grace M.       18
Jean B.        17
Claudine R.    17
Eric N.        16
Patrick K.     16
Aline U.       16
Unassigned      7
Name: count, dtype: int64


### 🧩 Practice 2.2
Now that `sales_rep` values are clean, create a new boolean column `is_unassigned` that is `True`
wherever `sales_rep` equals `"Unassigned"` and `False` otherwise. Then, separately, create a column
`region_rep` that combines region and rep into one label, formatted like `"East - Grace T."`
(a string that concatenates the `region` and `sales_rep` columns with `" - "` in between).

**Hints:**
- A boolean column from a comparison: `df["col"] == "some_value"`.
- String columns can be joined with `+`: `df["a"] + " - " + df["b"]` (make sure both sides are
  already strings).

In [13]:
# TODO: create df_sales["is_unassigned"] as a boolean column
# TODO: create df_sales["region_rep"] combining region and sales_rep

## 2.3 Date parsing & formatting

**Concept:** `pd.to_datetime(series, errors="coerce")` tries to parse *any* reasonable date
representation into a real `datetime64` column, and turns anything it can't parse into `NaT`
(pandas' "missing datetime") instead of raising an error and killing your whole pipeline. Once a
column is a true datetime dtype, you get the `.dt` accessor for free: `.dt.month`, `.dt.day_name()`,
`.dt.strftime("%Y-%m")`, etc.

**Why it matters on the job:** Excel exports notoriously mix date formats (`8/5/2026`,
`05-Aug-2026`, `2026-08-05`, or plain broken text like `"TBD"`) within the *same column*. Silently
ignoring parse failures means silently dropping rows out of every date-based report later — so
always check how many failed and *why* before moving on.

In [14]:
# order_date is currently plain text in several different formats -- coerce it to real datetimes
df_sales["order_date"] = pd.to_datetime(df_sales["order_date"], errors="coerce")

n_failed = df_sales["order_date"].isna().sum()
print(f"Rows where the date failed to parse: {n_failed}")
print(df_sales_raw.loc[df_sales["order_date"].isna(), "Order Date "].unique())  # show WHAT failed

# Now that it's a real datetime column, derived fields are essentially free:
df_sales["order_month"] = df_sales["order_date"].dt.to_period("M").astype(str)
df_sales["order_weekday"] = df_sales["order_date"].dt.day_name()

df_sales[["order_date", "order_month", "order_weekday"]].head()

Rows where the date failed to parse: 101
<StringArray>
['17-Jun-2026',  '2026-04-13',  '2026-01-17',           nan, '13-May-2026',
 '11-Jun-2026', '27-May-2026',  '2026-04-29',  '2026-02-18',  '2026-06-02',
 '09-Mar-2026', '18-May-2026',  '2026-03-14', '12-Jun-2026', '10-Jan-2026',
  '2026-03-13', '29-May-2026',  '2026-01-12', '22-Mar-2026',  '2026-06-20',
  '2026-06-28',  '2026-05-02', '19-May-2026',  '2026-04-10', '21-Jan-2026',
  '2026-03-21', '15-Jun-2026',  '2026-05-18', '26-May-2026', '02-Feb-2026',
 '23-Apr-2026', '24-Apr-2026', '09-Jan-2026',  '2026-04-16',  '2026-01-01',
  '2026-04-23',  '2026-03-04', '10-May-2026',  '2026-05-15', '25-Jun-2026',
  '2026-04-08', '15-Mar-2026',  '2026-05-04', '08-May-2026',  '2026-01-25',
 '21-Feb-2026',  '2026-03-09',  '2026-06-08',  '2026-03-19', '26-Jan-2026',
  '2026-01-14',  '2026-06-29', '27-Apr-2026',  '2026-01-24', '01-Feb-2026',
  '2026-06-18', '18-Jun-2026', '06-Feb-2026',  '2026-01-05', '13-Feb-2026',
  '2026-05-25', '04-Jun-2026',  '

,order_date,order_month,order_weekday
0,2026-02-26,2026-02,Thursday
1,NaT,NaN,NaN
2,NaT,NaN,NaN
3,NaT,NaN,NaN
4,NaT,NaN,NaN


### 🧩 Practice 2.3
Create a new column `is_weekend` that is `True` when `order_weekday` is `"Saturday"` or
`"Sunday"`, and `False` otherwise (rows with a failed/missing date should end up `False`, not
`NaN` — think about how `.isin()` handles missing values).

**Hints:**
- `series.isin(["Saturday", "Sunday"])` returns a boolean Series directly.
- `.isin()` naturally returns `False` for `NaN` input, which is exactly the behavior we want here.

In [15]:
# TODO: create df_sales["is_weekend"]

## 2.4 Type casting

**Concept:** `pd.to_numeric(series, errors="coerce")` is the numeric sibling of `to_datetime` —
it forces a column to a real numeric dtype and turns unparseable entries into `NaN` instead of
crashing. From there, `.fillna()` lets you decide *deliberately* how to handle the gaps (zero?
median? drop the row?) rather than letting pandas' `object` dtype quietly turn `5` into `"5"`
somewhere in a merge or a `.sum()`.

**Why it matters on the job:** A "numeric" column that's secretly `dtype: object` (because of one
stray blank cell) will make `.sum()` either throw an error or silently concatenate strings instead
of adding numbers — a bug that's brutal to spot after the fact.

In [16]:
print("Qty dtype before:", df_sales["qty"].dtype)

df_sales["qty"] = pd.to_numeric(df_sales["qty"], errors="coerce")
print("Qty dtype after:", df_sales["qty"].dtype)
print("Missing qty values:", df_sales["qty"].isna().sum())

# Business decision: a missing quantity on a logged order most likely means "1 unit" (the
# minimum possible order), not zero -- document *why* you chose this, don't just fillna(0) blindly
df_sales["qty"] = df_sales["qty"].fillna(1).astype(int)

df_sales["qty"].describe()

Qty dtype before: float64
Qty dtype after: float64
Missing qty values: 11


count    150.000000
mean       5.520000
std        3.016109
min        1.000000
25%        3.000000
50%        6.000000
75%        8.000000
max       10.000000
Name: qty, dtype: float64

### 🧩 Practice 2.4
`df_sales["order_id"]` looks numeric-ish (`"ORD-1000"`) but is really a text identifier — it
should **stay** a string/object dtype (casting IDs to numbers is a
classic mistake: you lose leading structure and can't do lookups reliably). Instead, practice the
*reverse* skill: cast the `"customer_id"` column to pandas' nullable `"string"` dtype explicitly
using `.astype("string")`, and confirm the dtype changed.

**Hints:**
- `.astype("string")` uses pandas' nullable string dtype (capital-S aware), distinct from generic
  Python `object` dtype — useful because it plays more predictably with `.isna()`.
- Check the change with `df_sales["customer_id"].dtype`.

In [17]:
# TODO: cast df_sales["customer_id"] to the "string" dtype

## 2.5 Calculated columns

**Concept:** Vectorized arithmetic (`df["a"] * df["b"]`) is almost always faster and clearer than
`.apply()` with a Python function — reserve `.apply()` for logic that genuinely can't be expressed
with built-in vectorized operations. `np.where(condition, if_true, if_false)` is the vectorized
equivalent of an Excel `IF()` formula, and is great for flagging or bucketing.

**Why it matters on the job:** This is exactly what "adding a calculated column" means in Excel —
except it runs on 150 rows or 15 million rows in about the same amount of code.

In [18]:
import numpy as np

# A simple flag: is this a bulk order? (qty of 5 or more)
df_sales["is_bulk_order"] = df_sales["qty"] >= 5

# A bucketed column using np.where -- the vectorized equivalent of Excel's IF()
df_sales["order_size"] = np.where(df_sales["qty"] >= 5, "Bulk",
                          np.where(df_sales["qty"] >= 3, "Medium", "Small"))

df_sales[["qty", "is_bulk_order", "order_size"]].sample(6, random_state=1)

,qty,is_bulk_order,order_size
14,5,True,Bulk
98,1,False,Small
75,9,True,Bulk
16,1,False,Small
131,10,True,Bulk
56,5,True,Bulk


### 🧩 Practice 2.5
Create a column `days_since_start` equal to the number of days between each order's `order_date`
and the earliest date in the whole column (`df_sales["order_date"].min()`), as a plain integer.
Rows with a missing `order_date` should end up with a missing (`NaN`) result — that's fine, leave
them as-is.

**Hints:**
- Subtracting two datetime Series gives a `Timedelta` Series; `.dt.days` converts that to an integer
  count of days.
- `df_sales["order_date"].min()` ignores `NaT` automatically.

In [19]:
# TODO: create df_sales["days_since_start"]

### ✅ Checkpoint
Run the cell below to confirm `df_sales` is in good shape before moving on to Module 3. If any
assertion fails, scroll back up — one of the worked examples (not the practice cells) needs to have
been run in order.

In [20]:
assert df_sales["qty"].dtype.kind in "iu", "qty should be an integer dtype by now"
assert str(df_sales["order_date"].dtype).startswith("datetime64"), "order_date should be a real datetime"
assert set(df_sales["region"].unique()) <= {"North", "South", "East", "West"}, "region should be normalized"
print("✅ df_sales looks clean. Columns:", df_sales.columns.tolist())
df_sales.head()

✅ df_sales looks clean. Columns: ['order_id', 'order_date', 'customer_id', 'product_code', 'qty', 'region', 'sales_rep', 'notes', 'order_month', 'order_weekday', 'is_bulk_order', 'order_size']


,order_id,order_date,customer_id,product_code,qty,region,sales_rep,notes,order_month,order_weekday,is_bulk_order,order_size
0,ORD-1000,2026-02-26,C002,P013,6,East,Diane I.,Rush order,2026-02,Thursday,True,Bulk
1,ORD-1001,NaT,C005,P005,8,South,Diane I.,No note,NaN,NaN,True,Bulk
2,ORD-1002,NaT,C016,P002,6,North,Grace M.,No note,NaN,NaN,True,Bulk
3,ORD-1003,NaT,C009,P009,7,North,Diane I.,No note,NaN,NaN,True,Bulk
4,ORD-1004,NaT,C024,P005,7,South,Eric N.,Follow up needed,NaN,NaN,True,Bulk


---
# Module 3: Advanced Operations

With a clean `df_sales`, we can now do the analytical heavy lifting: pulling in reference data
(the pandas answer to VLOOKUP/XLOOKUP), summarizing with pivot tables and `groupby`, and using
multi-level indexes to slice results the way you would with a PivotTable's row/column fields in
Excel.

## 3.1 VLOOKUP / XLOOKUP → `pd.merge`

**Concept:** Excel's `VLOOKUP`/`XLOOKUP` pull a value from a lookup table into your main table based
on a matching key. `pd.merge(left, right, on="key", how="left")` does the same thing for the
*entire table at once* — every row in `left` gets matching columns from `right` wherever the key
lines up, and `how="left"` (like XLOOKUP's default) keeps every row from `left` even if there's no
match, filling those spots with `NaN` (XLOOKUP's `if_not_found` argument).

**Why it matters on the job:** A single `merge()` call replaces dragging a `VLOOKUP` formula down
thousands of rows — and it's dramatically less error-prone, since there's no risk of a formula's
range reference silently drifting.

| Excel | pandas |
|---|---|
| `VLOOKUP(key, table, col_index, FALSE)` | `df.merge(lookup_table, on="key", how="left")` |
| `XLOOKUP(key, lookup_range, return_range, "Not found")` | `merge(...)` then `.fillna("Not found")` |

In [21]:
# Load the two clean reference/master tables
customers = pd.read_excel("company_data.xlsx", sheet_name="Customers")
products = pd.read_excel("company_data.xlsx", sheet_name="Products")

# Normalize their column names too, for consistency with df_sales
customers.columns = customers.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
products.columns = products.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
customers = customers.rename(columns={"custid": "customer_id"})
products = products.rename(columns={"productcode": "product_code"})

# LEFT JOIN both lookup tables onto df_sales, exactly like two VLOOKUPs / XLOOKUPs chained together
df_merged = (
    df_sales
    .merge(products, on="product_code", how="left")
    .merge(customers, on="customer_id", how="left")
)

# XLOOKUP-style "if not found" handling: flag + fill any rows that didn't match a product or customer
unmatched_products = df_merged["productname"].isna().sum() if "productname" in df_merged else df_merged["product_name"].isna().sum()
print("Orders with no matching product record:", unmatched_products)

# Calculated column that ONLY became possible after the lookup: revenue per line item
df_merged["total_amount"] = df_merged["qty"] * df_merged["unitcost"] if "unitcost" in df_merged else df_merged["qty"] * df_merged["unit_cost"]

df_merged.head()

Orders with no matching product record: 0


,order_id,order_date,customer_id,product_code,qty,region,sales_rep,notes,order_month,order_weekday,is_bulk_order,order_size,productname,category,unitcost,customername,segment,country,total_amount
0,ORD-1000,2026-02-26,C002,P013,6,East,Diane I.,Rush order,2026-02,Thursday,True,Bulk,Whiteboard Markers,Office Supplies,4.5,Nyanza Traders Ltd,Enterprise,Kenya,27.0
1,ORD-1001,NaT,C005,P005,8,South,Diane I.,No note,NaN,NaN,True,Bulk,Mechanical Keyboard,Electronics,89.0,Rift Valley Logistics,Startup,Kenya,712.0
2,ORD-1002,NaT,C016,P002,6,North,Grace M.,No note,NaN,NaN,True,Bulk,Wireless Mouse,Electronics,24.5,Ivory Logistics,Startup,Rwanda,147.0
3,ORD-1003,NaT,C009,P009,7,North,Diane I.,No note,NaN,NaN,True,Bulk,Desk Lamp,Furniture,45.0,Highland Tech Solutions,Startup,Tanzania,315.0
4,ORD-1004,NaT,C024,P005,7,South,Eric N.,Follow up needed,NaN,NaN,True,Bulk,Mechanical Keyboard,Electronics,89.0,Ridge Logistics,SMB,Uganda,623.0


### 🧩 Practice 3.1
Some orders reference a `customer_id` that (in a real system) might have been deleted from the
customer master. Count how many rows in `df_merged` have a missing `segment` (meaning the customer
lookup didn't find a match), and store the count in `unmatched_customers`. Then create a new
column `customer_name_clean` that's equal to `customername` (or `customer_name`, check the actual
column) wherever it's known, and the text `"Unknown Customer"` wherever it's missing — the pandas
equivalent of `XLOOKUP(..., if_not_found="Unknown Customer")`.

**Hints:**
- Run `df_merged.columns.tolist()` first to see the exact column name pandas gave the customer name
  field after the merge (merges sometimes add suffixes if names collide).
- `.fillna("some text")` is the direct equivalent of XLOOKUP's `if_not_found` argument.

In [22]:
# TODO: unmatched_customers = ...
unmatched_customers = None  # <- replace this

# TODO: df_merged["customer_name_clean"] = ...

## 3.2 Pivot tables

**Concept:** `pd.pivot_table()` is a direct code equivalent of Excel's PivotTable UI: `index=`
sets the row field(s), `columns=` sets the column field(s), `values=` sets what's being aggregated,
and `aggfunc=` sets how (`"sum"`, `"mean"`, `"count"`, or a list of several at once). `margins=True`
adds the "Grand Total" row/column you'd get from checking that box in Excel.

**Why it matters on the job:** Rebuilding an Excel PivotTable in code means the whole summary can be
regenerated automatically every time new data lands — no more re-dragging fields after every
refresh.

In [23]:
pivot_region_category = pd.pivot_table(
    df_merged,
    values="total_amount",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,     # cells with no orders show 0, not NaN -- matches Excel PivotTable behavior
    margins=True,      # adds a "All" row and column, like Excel's Grand Total
    margins_name="Total",
)
pivot_region_category.round(2)

category,Electronics,Furniture,Office Supplies,Total
region,,,,
East,33626.32,16124.0,528.05,50278.37
North,27314.00,13082.0,334.35,40730.35
South,5042.50,5910.0,225.00,11177.50
West,21759.39,13508.0,263.55,35530.94
Total,87742.21,48624.0,1350.95,137717.16


### 🧩 Practice 3.2
Build a pivot table called `pivot_rep_month` showing **total quantity sold** (`qty`), with
`sales_rep` as rows and `order_month` as columns, using `aggfunc="sum"` and filling empty cells
with `0`.

**Hints:**
- Same `pd.pivot_table()` shape as the worked example, just swap `values`, `index`, and `columns`.
- Don't forget `fill_value=0`.

In [24]:
# TODO: pivot_rep_month = pd.pivot_table(...)
pivot_rep_month = None  # <- replace this

## 3.3 Grouping & aggregation

**Concept:** `.groupby("col").agg(...)` is more flexible than `pivot_table` when you want several
*differently-named* summary columns at once (e.g. total revenue **and** order count **and** average
order size, all in a single readable result) rather than one value spread across a grid.
"Named aggregation" — `agg(new_col=("source_col", "func"))` — lets you name each output column
directly instead of dealing with pandas' default multi-level column names.

**Why it matters on the job:** This is the pattern behind almost every "summary by X" business
question: total sales by region, order count by rep, average basket size by segment, etc.

In [25]:
rep_summary = df_merged.groupby("sales_rep").agg(
    total_revenue=("total_amount", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_amount", "mean"),
).round(2).sort_values("total_revenue", ascending=False)

rep_summary

,total_revenue,order_count,avg_order_value
sales_rep,,,
Yves T.,24982.29,19,1314.86
Patrick K.,23093.25,16,1443.33
Diane I.,20713.05,24,863.04
Unassigned,18662.50,7,2666.07
Eric N.,13520.70,16,845.04
Aline U.,10136.32,16,633.52
Claudine R.,9990.35,17,587.67
Jean B.,9785.30,17,575.61
Grace M.,6833.40,18,379.63


### 🧩 Practice 3.3
Build `region_segment_summary`: group `df_merged` by **both** `region` and `segment` at once, and
compute `total_revenue` (sum of `total_amount`) and `num_orders` (count of `order_id`) for each
combination, sorted by `total_revenue` descending.

**Hints:**
- `.groupby(["col_a", "col_b"])` groups by more than one column simultaneously.
- Use the same named-aggregation `.agg(new_name=("source_col", "func"))` pattern shown above.

In [26]:
# TODO: region_segment_summary = df_merged.groupby([...]).agg(...)
region_segment_summary = None  # <- replace this

## 3.4 Multi-indexing

**Concept:** Grouping by two or more columns produces a **MultiIndex** — a row index with more than
one level, just like the nested row groups you get from Excel's PivotTable when you drop two fields
into the Rows area. `.loc[]` with a tuple selects a specific combination; `.unstack()` pivots an
inner index level out into columns (turning a "tall" grouped result into a "wide" grid, exactly what
Excel does automatically when you drag a second field into Columns instead of Rows).

**Why it matters on the job:** MultiIndex results are what `groupby([...])` naturally gives you —
understanding `.loc`, `.unstack()`, and `.reset_index()` means you're never stuck fighting the
output shape to get the table format you actually need.

In [27]:
region_category_multi = df_merged.groupby(["region", "category"])["total_amount"].sum()
print("MultiIndex Series:")
print(region_category_multi.head(8))

# Select one specific region with .loc -- just like filtering to one PivotTable row group
print("\nJust the 'East' region:")
print(region_category_multi.loc["East"])

# .unstack() turns the inner index level (category) into columns -- a "tall" result becomes "wide"
print("\nUnstacked into a region x category grid:")
region_category_multi.unstack(fill_value=0).round(2)

MultiIndex Series:
region  category       
East    Electronics        33626.32
        Furniture          16124.00
        Office Supplies      528.05
North   Electronics        27314.00
        Furniture          13082.00
        Office Supplies      334.35
South   Electronics         5042.50
        Furniture           5910.00
Name: total_amount, dtype: float64

Just the 'East' region:
category
Electronics        33626.32
Furniture          16124.00
Office Supplies      528.05
Name: total_amount, dtype: float64

Unstacked into a region x category grid:


category,Electronics,Furniture,Office Supplies
region,,,
East,33626.32,16124.0,528.05
North,27314.00,13082.0,334.35
South,5042.50,5910.0,225.00
West,21759.39,13508.0,263.55


### 🧩 Practice 3.4
Starting from `region_category_multi`, use `.loc[]` with a tuple to retrieve the single total
revenue value for `region="West"`, `category="Furniture"`, and store it in `west_furniture_total`.
Separately, call `.reset_index()` on `region_category_multi` and store the resulting flat DataFrame
(three plain columns instead of a MultiIndex Series) in `region_category_flat`.

**Hints:**
- A tuple selector looks like `series.loc[("West", "Furniture")]`.
- `.reset_index()` turns index levels back into regular columns and gives the Series a default
  integer index — very useful right before exporting to Excel, since flat tables write more
  predictably than MultiIndex ones.

In [28]:
# TODO: west_furniture_total = region_category_multi.loc[(...)]
west_furniture_total = None  # <- replace this

# TODO: region_category_flat = region_category_multi.reset_index()
region_category_flat = None  # <- replace this

---
# Module 4: Writing Back & Excel Formatting

The last mile of most real analyses: getting your clean, summarized results back into a polished
`.xlsx` file a non-technical stakeholder can open directly — multiple sheets, styled headers, live
formulas, and a chart, all generated automatically instead of built by hand every reporting cycle.

## 4.1 Exporting to multiple sheets

**Concept:** `pd.ExcelWriter(path, engine="openpyxl")` used as a context manager lets you write
several DataFrames to **different sheets of the same workbook** — each `.to_excel(writer,
sheet_name=...)` call adds one more sheet, and the file is only finalized when the `with` block
exits.

**Why it matters on the job:** Stakeholders usually want one workbook with a `Raw Data` tab, a
`Summary` tab, and maybe a `Chart` tab — not five separate files.

In [29]:
report_path = "sales_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    df_merged.to_excel(writer, sheet_name="Detail", index=False)
    rep_summary.reset_index().to_excel(writer, sheet_name="By Rep", index=False)
    pivot_region_category.to_excel(writer, sheet_name="Region x Category")

print(f"Wrote {report_path} with sheets: Detail, By Rep, Region x Category")

Wrote sales_report.xlsx with sheets: Detail, By Rep, Region x Category


### 🧩 Practice 4.1
Add a **fourth sheet** called `"By Month"` to `sales_report.xlsx`, containing total revenue per
`order_month` (you'll need a quick `groupby` first). You can either re-run a full `ExcelWriter`
block that includes all four sheets, or use `mode="a"` (append) with
`if_sheet_exists="replace"` to add just the one new sheet to the existing file.

**Hints:**
- `pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace")` opens an
  *existing* file to add/replace sheets rather than overwriting the whole workbook.
- `df.groupby("order_month")["total_amount"].sum()` gets you most of the way there.

In [30]:
# TODO: build a monthly revenue summary and write it to a new "By Month" sheet in sales_report.xlsx

## 4.2 Custom cell styles: fonts, fills, borders, widths

**Concept:** Once a workbook exists, `openpyxl.load_workbook()` opens it for editing at the cell
level. `openpyxl.styles` gives you `Font` (bold, color, size), `PatternFill` (background color),
`Alignment`, and `Border` objects that you assign directly to `cell.font`, `cell.fill`, etc.
Column widths live on `ws.column_dimensions["A"].width` rather than on individual cells.

**Why it matters on the job:** A bolded, colored header row is a small thing, but it's the
difference between a report that looks automatically generated and one that looks intentionally
built for a human reader.

In [31]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = load_workbook(report_path)
ws = wb["By Rep"]

# Style the header row: bold white text on a dark blue fill, centered
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_align = Alignment(horizontal="center", vertical="center")
thin_border = Border(*[Side(style="thin", color="B7B7B7")] * 4)

for cell in ws[1]:  # ws[1] is the entire first row
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_align

# Auto-ish column widths based on the longest value in each column (a common, simple heuristic)
for col_cells in ws.iter_cols(min_row=1, max_row=ws.max_row):
    col_letter = col_cells[0].column_letter
    longest = max(len(str(c.value)) for c in col_cells if c.value is not None)
    ws.column_dimensions[col_letter].width = longest + 4

# Light borders on every data cell, for a finished look
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for cell in row:
        cell.border = thin_border

wb.save(report_path)
print("Styled the 'By Rep' sheet header, widths, and borders.")

Styled the 'By Rep' sheet header, widths, and borders.


### 🧩 Practice 4.2
On the same `"By Rep"` sheet, apply a **currency number format** (`'#,##0.00'`) to the
`total_revenue` and `avg_order_value` columns (columns B and D, if `sales_rep` is column A) for
every data row. Number formats in openpyxl are set via `cell.number_format = "..."`.

**Hints:**
- Loop over `ws.iter_rows(min_row=2, max_row=ws.max_row)` to skip the header row.
- `cell.number_format = "#,##0.00"` is the openpyxl equivalent of Excel's Format Cells → Number.
- Remember to `wb.save(report_path)` again at the end, or your formatting won't persist.

In [32]:
# TODO: apply a currency number format to the total_revenue and avg_order_value columns

## 4.3 Adding live Excel formulas

**Concept:** openpyxl writes whatever string you put in a cell — if that string starts with `=`,
Excel will treat it as a live formula and calculate it when the file is opened, exactly as if you'd
typed it by hand. This means you can generate formulas that reference specific cells/ranges
programmatically (e.g. `f"=SUM(B2:B{last_row})"`), which is powerful for building totals rows that
stay "live" and recalculate if a stakeholder edits the raw numbers.

**Why it matters on the job:** Sometimes you *want* the recipient to see a real formula (so they
trust the number, or so it updates if they change an input) rather than a pandas-computed static
value baked in at export time.

In [33]:
ws = wb["By Rep"]
last_row = ws.max_row
total_row = last_row + 1

# Write a live SUM formula under the total_revenue column, and a live AVERAGE formula
# under the avg_order_value column -- these will recalculate in Excel if the data changes
ws.cell(row=total_row, column=1, value="TOTAL")
ws.cell(row=total_row, column=2, value=f"=SUM(B2:B{last_row})")
ws.cell(row=total_row, column=4, value=f"=AVERAGE(D2:D{last_row})")

for cell in ws[total_row]:
    cell.font = Font(bold=True)

wb.save(report_path)
print(f"Added a formula-driven TOTAL row at Excel row {total_row}.")

Added a formula-driven TOTAL row at Excel row 11.


### 🧩 Practice 4.3
On the `"Detail"` sheet, add a formula-driven total for the `total_amount` column, in the same
style as the example above (a `"TOTAL"` label in column A of the row right after the last data row,
and a `=SUM(...)` formula in the correct column for `total_amount`).

**Hints:**
- You'll need to find *which* column letter/number `total_amount` landed in — `ws[1]` (the header
  row) lets you search for the cell whose `.value == "total_amount"` and read its `.column`.
- `ws.max_row` still gives you the last row with data, same idea as the worked example.

In [34]:
# TODO: add a formula-driven TOTAL row for total_amount on the "Detail" sheet

## 4.4 Creating charts

**Concept:** `openpyxl.chart` builds native, editable Excel charts (not static images) — you define
a `Reference` (a cell range) for your categories and data, attach them to a chart object
(`BarChart`, `LineChart`, `PieChart`, ...), and add the chart to a worksheet with `ws.add_chart()`.
The chart stays linked to the cells, so if the underlying numbers change, the chart updates too —
just like a chart you built by hand in Excel.

**Why it matters on the job:** A native Excel chart (rather than a pasted-in image) is something the
recipient can click into, tweak, and copy into a PowerPoint deck — much more useful in a business
setting.

In [35]:
from openpyxl.chart import BarChart, Reference

ws = wb["By Rep"]

chart = BarChart()
chart.title = "Total Revenue by Sales Rep"
chart.x_axis.title = "Sales Rep"
chart.y_axis.title = "Revenue"
chart.style = 10

# Reference ranges point at the cells to plot -- data (with header, for the series name)
# and categories (the labels along the x-axis)
data = Reference(ws, min_col=2, min_row=1, max_row=last_row)       # total_revenue column, incl. header
cats = Reference(ws, min_col=1, min_row=2, max_row=last_row)       # sales_rep column, no header

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
chart.width, chart.height = 18, 10

ws.add_chart(chart, "F2")  # anchor the chart's top-left corner at cell F2
wb.save(report_path)
print("Added a native Excel bar chart to the 'By Rep' sheet.")

Added a native Excel bar chart to the 'By Rep' sheet.


### 🧩 Bonus: the same idea with `xlsxwriter`

**Concept:** `xlsxwriter` is a **write-only** library — it can create brand-new `.xlsx` files with
excellent formatting/chart support in a single pass, but (unlike openpyxl) it **cannot open or edit
an existing file**. The trade-off: for a report you build fresh every run (rather than incrementally
edit), `xlsxwriter`'s API is often a little more concise for combining formatting and charts.

**Why it matters on the job:** Knowing both libraries means picking the right tool: openpyxl when
you need to *read or incrementally modify* a workbook, xlsxwriter when you're *generating a report
from scratch* every time and want compact formatting + chart code.

In [36]:
import xlsxwriter

with xlsxwriter.Workbook("xlsxwriter_demo.xlsx") as xwb:
    xws = xwb.add_worksheet("By Rep")

    # Formats are defined once and reused -- a different style from openpyxl's per-cell objects
    header_fmt = xwb.add_format({"bold": True, "font_color": "white", "bg_color": "#1F4E78", "align": "center"})
    money_fmt = xwb.add_format({"num_format": "#,##0.00"})

    rep_summary_reset = rep_summary.reset_index()
    xws.write_row(0, 0, rep_summary_reset.columns.tolist(), header_fmt)
    for r, row in enumerate(rep_summary_reset.itertuples(index=False), start=1):
        xws.write(r, 0, row.sales_rep)
        xws.write(r, 1, row.total_revenue, money_fmt)
        xws.write(r, 2, row.order_count)
        xws.write(r, 3, row.avg_order_value, money_fmt)

    xws.set_column(0, 0, 18)
    xws.set_column(1, 3, 16)

    chart = xwb.add_chart({"type": "column"})
    n = len(rep_summary_reset)
    chart.add_series({
        "name": "Total Revenue",
        "categories": ["By Rep", 1, 0, n, 0],
        "values": ["By Rep", 1, 1, n, 1],
    })
    chart.set_title({"name": "Total Revenue by Sales Rep (xlsxwriter)"})
    xws.insert_chart("F2", chart)

print("Wrote xlsxwriter_demo.xlsx (built from scratch in a single pass).")

Wrote xlsxwriter_demo.xlsx (built from scratch in a single pass).


### 🧩 Practice 4.4
Using **openpyxl**, add a `LineChart` (import from `openpyxl.chart`) to a new sheet in
`sales_report.xlsx` showing total revenue by `order_month` (reuse the monthly summary you built in
Practice 4.1, or rebuild it quickly). Anchor it at cell `"B2"`.

**Hints:**
- The pattern is identical to the `BarChart` example — build a `Reference` for data and one for
  categories, `chart.add_data(...)`, `chart.set_categories(...)`, `ws.add_chart(chart, "B2")`.
- `from openpyxl.chart import LineChart, Reference`

In [37]:
# TODO: add a LineChart of monthly revenue to sales_report.xlsx

## 4.5 Bonus: conditional formatting

**Concept:** `openpyxl.formatting.rule` provides rule objects that mirror Excel's Conditional
Formatting menu: `ColorScaleRule` (a 2 or 3-color heatmap), `CellIsRule` (highlight cells that
meet a comparison), and others. You attach a rule to a range with
`ws.conditional_formatting.add(range_string, rule)`.

**Why it matters on the job:** A heatmap over a revenue table lets a reader spot the best/worst
performers at a glance — genuinely useful, and something stakeholders often specifically ask for.

In [38]:
from openpyxl.formatting.rule import ColorScaleRule

ws = wb["By Rep"]
color_scale = ColorScaleRule(
    start_type="min", start_color="F8696B",     # red for the lowest values
    mid_type="percentile", mid_value=50, mid_color="FFEB84",  # yellow at the midpoint
    end_type="max", end_color="63BE7B",          # green for the highest values
)
ws.conditional_formatting.add(f"B2:B{last_row}", color_scale)

wb.save(report_path)
print("Added a red-yellow-green heatmap to the total_revenue column.")

Added a red-yellow-green heatmap to the total_revenue column.


### 🧩 Practice 4.5
Add a `CellIsRule` to the `"Detail"` sheet's `total_amount` column that fills any cell with a value
**greater than 1000** in a light red/pink so unusually large line items jump out visually.

**Hints:**
- `from openpyxl.formatting.rule import CellIsRule` and, separately,
  `from openpyxl.styles import PatternFill` for the fill to apply.
- `CellIsRule(operator="greaterThan", formula=["1000"], fill=your_fill)`
- You'll need to find `total_amount`'s column letter on `"Detail"` first, same idea as Practice 4.3.

In [39]:
# TODO: add conditional formatting highlighting total_amount > 1000 on the "Detail" sheet

---
# 🏆 Capstone Challenge: Build an Executive Report From Scratch

Time to put every module together without a worked example to lean on. Using only
`company_data.xlsx` (the original messy file from the Setup cell), build a polished
**`executive_report.xlsx`** workbook, completely from scratch, in the cell below.

### Requirements checklist
1. **Load & inspect** — read the `RawExport` sheet correctly (mind the banner rows!), plus the
   `Customers` and `Products` reference sheets.
2. **Clean** — normalize column names, fix text casing, parse dates properly, cast `qty` to a
   real integer, and handle missing values deliberately (don't just silently drop rows).
3. **Merge** — join in product and customer details (the VLOOKUP/XLOOKUP equivalent), and compute
   `total_amount` per line item.
4. **Summarize** — build a `groupby`/pivot summary of total revenue by `region` **and** by
   `category`.
5. **Export & format** — write `executive_report.xlsx` with at least:
   - A `"Summary by Region"` sheet with a styled (bold/colored) header row and a formula-driven
     `TOTAL` row.
   - A `"Summary by Category"` sheet with a native Excel chart.
   - Conditional formatting somewhere that highlights values above or below a threshold you choose.

Work top-to-bottom through the requirements — you already have every piece of code you need
scattered through Modules 1–4 above. The point of a capstone is *assembling* it, not
memorizing new syntax.

In [40]:
# TODO: Capstone -- build executive_report.xlsx from scratch here.
# Feel free to use as many cells as you like; this is just a starting point.

# 1. Load & inspect
# 2. Clean
# 3. Merge
# 4. Summarize
# 5. Export & format

---
# 📎 Solutions Appendix

**Spoilers below.** Solutions for every Practice Challenge, in order. Try each challenge yourself
first — these are here so you can check your work or get unstuck, not to be read first.

Each solution cell is self-contained where practical (it re-derives anything it needs), so you can
run an individual solution cell without necessarily having run every cell above it in order —
though running the whole notebook top to bottom will always work too.

## Module 1 solutions

In [41]:
# --- 1.1 ---
df_products_check = pd.read_excel("company_data.xlsx", sheet_name="Products")
print(df_products_check.shape)
print(df_products_check.columns.tolist())

(15, 4)
['ProductCode', 'ProductName', 'Category', 'UnitCost']


In [42]:
# --- 1.2 ---
raw_peek_sol = pd.read_excel("company_data.xlsx", sheet_name="RawExport", header=None, nrows=5)
print(raw_peek_sol)
# Row 0 (physical row 1) is the banner text, row 1 (physical row 2) is blank,
# row 2 (physical row 3) is the real header -- so we need to skip 2 rows.
discovered_skiprows = 2
print("discovered_skiprows =", discovered_skiprows)

                                                   0            1        2  \
0  Company Sales Export -- Generated by FinanceSy...          NaN      NaN   
1                                                NaN          NaN      NaN   
2                                            OrderID  Order Date   CustID    
3                                           ORD-1000   02/26/2026     C002   
4                                           ORD-1001  17-Jun-2026     C005   

              3    4         5           6           7  
0           NaN  NaN       NaN         NaN         NaN  
1           NaN  NaN       NaN         NaN         NaN  
2   ProductCode  Qty   Region     SalesRep       Notes  
3          P013    6      East  Diane I.    Rush order  
4          P005    8     SOUTH    DIANE I.         NaN  


discovered_skiprows = 2


In [43]:
# --- 1.3 ---
qty_dtype = df_sales_raw["Qty"].dtype
region_unique_count = df_sales_raw[" Region "].nunique()
print("qty_dtype:", qty_dtype)
print("region_unique_count:", region_unique_count)  # inflated by whitespace/casing variants -- that's the point!

qty_dtype: float64
region_unique_count: 20


In [44]:
# --- 1.4 ---
pct_missing_qty = df_sales_raw["Qty"].isna().mean() * 100
print(f"pct_missing_qty: {pct_missing_qty:.2f}%")

pct_missing_qty: 7.33%


## Module 2 solutions

In [45]:
# --- 2.1 ---
df_sales_practice = df_sales.rename(columns={"notes": "order_notes"})
print(df_sales_practice.columns.tolist())

['order_id', 'order_date', 'customer_id', 'product_code', 'qty', 'region', 'sales_rep', 'order_notes', 'order_month', 'order_weekday', 'is_bulk_order', 'order_size']


In [46]:
# --- 2.2 ---
df_sales["is_unassigned"] = df_sales["sales_rep"] == "Unassigned"
df_sales["region_rep"] = df_sales["region"] + " - " + df_sales["sales_rep"]
df_sales[["region", "sales_rep", "is_unassigned", "region_rep"]].sample(5, random_state=2)

,region,sales_rep,is_unassigned,region_rep
6,West,Grace M.,False,West - Grace M.
3,North,Diane I.,False,North - Diane I.
113,West,Grace M.,False,West - Grace M.
12,North,Patrick K.,False,North - Patrick K.
24,West,Diane I.,False,West - Diane I.


In [47]:
# --- 2.3 ---
df_sales["is_weekend"] = df_sales["order_weekday"].isin(["Saturday", "Sunday"])
df_sales[["order_weekday", "is_weekend"]].drop_duplicates().sort_values("order_weekday")

,order_weekday,is_weekend
6,Friday,False
15,Monday,False
22,Saturday,True
5,Sunday,True
0,Thursday,False
13,Tuesday,False
11,Wednesday,False
1,NaN,False


In [48]:
# --- 2.4 ---
df_sales["customer_id"] = df_sales["customer_id"].astype("string")
print(df_sales["customer_id"].dtype)

string


In [49]:
# --- 2.5 ---
df_sales["days_since_start"] = (df_sales["order_date"] - df_sales["order_date"].min()).dt.days
df_sales[["order_date", "days_since_start"]].head()

,order_date,days_since_start
0,2026-02-26,56.0
1,NaT,NaN
2,NaT,NaN
3,NaT,NaN
4,NaT,NaN


## Module 3 solutions

In [50]:
# --- 3.1 ---
unmatched_customers = df_merged["segment"].isna().sum()
df_merged["customer_name_clean"] = df_merged["customername"].fillna("Unknown Customer")
print("unmatched_customers:", unmatched_customers)
df_merged[["customer_id", "customer_name_clean"]].head()

unmatched_customers: 0


,customer_id,customer_name_clean
0,C002,Nyanza Traders Ltd
1,C005,Rift Valley Logistics
2,C016,Ivory Logistics
3,C009,Highland Tech Solutions
4,C024,Ridge Logistics


In [51]:
# --- 3.2 ---
pivot_rep_month = pd.pivot_table(
    df_merged, values="qty", index="sales_rep", columns="order_month",
    aggfunc="sum", fill_value=0,
)
pivot_rep_month

order_month,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
sales_rep,,,,,,
Aline U.,0,0,2,0,4,7
Claudine R.,0,17,0,5,4,5
Diane I.,8,13,0,3,0,0
Eric N.,9,5,5,9,2,0
Grace M.,2,0,12,10,0,15
Jean B.,19,10,12,0,10,3
Patrick K.,0,18,12,0,0,5
Unassigned,0,0,0,5,23,0
Yves T.,2,0,7,7,0,0


In [52]:
# --- 3.3 ---
region_segment_summary = (
    df_merged.groupby(["region", "segment"])
    .agg(total_revenue=("total_amount", "sum"), num_orders=("order_id", "count"))
    .sort_values("total_revenue", ascending=False)
)
region_segment_summary

total_revenue  num_orders
region segment                              
North  Startup          23707.50           8
West   Startup          16475.99          17
East   Enterprise       13521.48          22
       SMB              12748.41           5
       Government       12524.50           8
       Startup          11483.98          14
North  Government       10761.10           8
West   Government        8553.00           5
       Enterprise        7837.90          12
South  Startup           3776.15          10
North  SMB               3753.00           4
South  SMB               2924.55          10
West   SMB               2664.05           7
South  Enterprise        2560.60           4
North  Enterprise        2508.75          10
South  Government        1916.20           6

In [53]:
# --- 3.4 ---
west_furniture_total = region_category_multi.loc[("West", "Furniture")]
region_category_flat = region_category_multi.reset_index()
print("west_furniture_total:", west_furniture_total)
region_category_flat.head()

west_furniture_total: 13508.0


,region,category,total_amount
0,East,Electronics,33626.32
1,East,Furniture,16124.00
2,East,Office Supplies,528.05
3,North,Electronics,27314.00
4,North,Furniture,13082.00


## Module 4 solutions

Each cell below reopens `sales_report.xlsx` fresh with `load_workbook()` before editing, so they're
safe to run individually without depending on in-memory state from earlier cells.

In [54]:
# --- 4.1 ---
monthly_summary = df_merged.groupby("order_month")["total_amount"].sum().reset_index()
with pd.ExcelWriter(report_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    monthly_summary.to_excel(writer, sheet_name="By Month", index=False)
print("Added 'By Month' sheet.")
monthly_summary

Added 'By Month' sheet.


,order_month,total_amount
0,2026-01,5455.5
1,2026-02,5559.0
2,2026-03,6419.7
3,2026-04,12162.0
4,2026-05,11083.2
5,2026-06,4726.5


In [55]:
# --- 4.2 ---
wb_sol = load_workbook(report_path)
ws_sol = wb_sol["By Rep"]
last_row_sol = ws_sol.max_row
for row in ws_sol.iter_rows(min_row=2, max_row=last_row_sol):
    row[1].number_format = "#,##0.00"  # total_revenue (column B)
    row[3].number_format = "#,##0.00"  # avg_order_value (column D)
wb_sol.save(report_path)
print("Applied currency formatting to 'By Rep'.")

Applied currency formatting to 'By Rep'.


In [56]:
# --- 4.3 ---
wb_sol = load_workbook(report_path)
ws_detail = wb_sol["Detail"]

header_cells = ws_detail[1]
col_idx = next(c.column for c in header_cells if c.value == "total_amount")
col_letter = ws_detail.cell(row=1, column=col_idx).column_letter

last_row_detail = ws_detail.max_row
total_row = last_row_detail + 1
ws_detail.cell(row=total_row, column=1, value="TOTAL")
ws_detail.cell(row=total_row, column=col_idx, value=f"=SUM({col_letter}2:{col_letter}{last_row_detail})")
for cell in ws_detail[total_row]:
    cell.font = Font(bold=True)

wb_sol.save(report_path)
print(f"Added TOTAL row at row {total_row}, column {col_letter}.")

Added TOTAL row at row 152, column S.


In [57]:
# --- 4.4 ---
from openpyxl.chart import LineChart, Reference as ChartReference

wb_sol = load_workbook(report_path)
ws_month = wb_sol["By Month"]
last_row_month = ws_month.max_row

line_chart = LineChart()
line_chart.title = "Total Revenue by Month"
data = ChartReference(ws_month, min_col=2, min_row=1, max_row=last_row_month)
cats = ChartReference(ws_month, min_col=1, min_row=2, max_row=last_row_month)
line_chart.add_data(data, titles_from_data=True)
line_chart.set_categories(cats)
ws_month.add_chart(line_chart, "D2")

wb_sol.save(report_path)
print("Added LineChart to 'By Month'.")

Added LineChart to 'By Month'.


In [58]:
# --- 4.5 ---
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import PatternFill as PF

wb_sol = load_workbook(report_path)
ws_detail = wb_sol["Detail"]

header_cells = ws_detail[1]
col_idx = next(c.column for c in header_cells if c.value == "total_amount")
col_letter = ws_detail.cell(row=1, column=col_idx).column_letter
last_row_detail = ws_detail.max_row

pink_fill = PF(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
rule = CellIsRule(operator="greaterThan", formula=["1000"], fill=pink_fill)
ws_detail.conditional_formatting.add(f"{col_letter}2:{col_letter}{last_row_detail}", rule)

wb_sol.save(report_path)
print(f"Highlighted {col_letter}2:{col_letter}{last_row_detail} where value > 1000.")

Highlighted S2:S152 where value > 1000.


## Capstone solution (compact reference)

This condenses every module into one pipeline. It's intentionally terse — in a real project you'd
likely keep it split into the cell-by-cell steps used throughout this notebook, which are much
easier to debug and review.

In [59]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.chart import BarChart, Reference
from openpyxl.formatting.rule import ColorScaleRule

# 1) Load & inspect
cap_sales = pd.read_excel("company_data.xlsx", sheet_name="RawExport", skiprows=2)
cap_customers = pd.read_excel("company_data.xlsx", sheet_name="Customers")
cap_products = pd.read_excel("company_data.xlsx", sheet_name="Products")

# 2) Clean
for df_ in (cap_sales, cap_customers, cap_products):
    df_.columns = df_.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
cap_sales = cap_sales.rename(columns={"custid": "customer_id", "productcode": "product_code",
                                       "orderid": "order_id", "salesrep": "sales_rep"})
cap_customers = cap_customers.rename(columns={"custid": "customer_id"})
cap_products = cap_products.rename(columns={"productcode": "product_code"})

cap_sales["region"] = cap_sales["region"].str.strip().str.title()
cap_sales["sales_rep"] = cap_sales["sales_rep"].str.strip().str.title().fillna("Unassigned")
cap_sales["order_date"] = pd.to_datetime(cap_sales["order_date"], errors="coerce")
cap_sales["qty"] = pd.to_numeric(cap_sales["qty"], errors="coerce").fillna(1).astype(int)

# 3) Merge
cap_merged = cap_sales.merge(cap_products, on="product_code", how="left").merge(cap_customers, on="customer_id", how="left")
cap_merged["total_amount"] = cap_merged["qty"] * cap_merged["unitcost"]

# 4) Summarize
summary_region = cap_merged.groupby("region")["total_amount"].sum().reset_index().sort_values("total_amount", ascending=False)
summary_category = cap_merged.groupby("category")["total_amount"].sum().reset_index().sort_values("total_amount", ascending=False)

# 5) Export & format
cap_path = "executive_report.xlsx"
with pd.ExcelWriter(cap_path, engine="openpyxl") as writer:
    summary_region.to_excel(writer, sheet_name="Summary by Region", index=False)
    summary_category.to_excel(writer, sheet_name="Summary by Category", index=False)

cap_wb = load_workbook(cap_path)

# Style + formula total row on "Summary by Region"
ws_r = cap_wb["Summary by Region"]
for cell in ws_r[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
last_r = ws_r.max_row
ws_r.cell(row=last_r + 1, column=1, value="TOTAL")
ws_r.cell(row=last_r + 1, column=2, value=f"=SUM(B2:B{last_r})").font = Font(bold=True)
ws_r.conditional_formatting.add(
    f"B2:B{last_r}",
    ColorScaleRule(start_type="min", start_color="F8696B", end_type="max", end_color="63BE7B"),
)

# Chart on "Summary by Category"
ws_c = cap_wb["Summary by Category"]
for cell in ws_c[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
last_c = ws_c.max_row
chart = BarChart()
chart.title = "Revenue by Category"
data = Reference(ws_c, min_col=2, min_row=1, max_row=last_c)
cats = Reference(ws_c, min_col=1, min_row=2, max_row=last_c)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)
ws_c.add_chart(chart, "D2")

cap_wb.save(cap_path)
print(f"Capstone complete -- wrote {cap_path}")

Capstone complete -- wrote executive_report.xlsx

---
## 🎓 Where to go from here

You now have working muscle memory for the four pillars of Excel automation in Python: **loading**
messy multi-sheet files correctly, **cleaning** text/dates/types deliberately, **joining and
summarizing** with `merge`/`pivot_table`/`groupby`, and **writing back** polished, formula-driven,
charted workbooks. A few natural next steps:

- Point this same pipeline at a real export from your own job and see what breaks — real data is
  always messier than a tutorial's.
- Look into `pandas.read_excel`'s `dtype=` and `converters=` arguments for even more control over
  how individual columns get parsed on load.
- Explore `openpyxl`'s support for pivot tables and slicers directly in the output file, or
  `xlsxwriter`'s support for sparklines, for even more Excel-native polish.